# Production plan with a distribution network

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ula-uab/lscm-decision-making/blob/main/cases/production_plan/notebooks/distribution_network.ipynb)

A company assembles electric bicycles in three plants and plans how many bikes each plant assembles each month from January to June. In the third version of the model, M2, the plants ship to a central warehouse, which holds stock and delivers the orders. With forecast 4, whose peak is in April and May, M2 has no solution: the bikes that have to be assembled in advance do not fit in the plants and the central warehouse. This notebook uses the fourth version, M3: a distribution network in Spain with the plants, the central warehouse, a rail terminal, two regional warehouses and four local warehouses, which receive the orders. The network is a list of arcs, each with a transport cost and an arc capacity.

Contents of the notebook:

1. Network, nodes and forecast 4.
2. Linear model of the distribution network.
3. Optimal plan under forecast 4.
4. Three tests, each with one datum changed: the arc capacity of the rail arc, the transport rate of rail and the node capacity of regional warehouse 2.

The cells run in order, with ▶ or *Shift + Enter*. Some cells have a form (sliders and drop-down lists): after changing a value, run the cell again. Outside Colab the form is a line such as `month = "May"`: edit the value and run the cell.

The table gives the symbols of M0 to M3, with the term that names each concept. The column «From» gives the version where each symbol first appears; in M3 the warehouses are nodes $i$ of the network.

| Symbol | Meaning | Term | Unit | From |
|---|---|---|---|---|
| $F$, $f$ | Plants | — | — | M0 |
| $T$, $t$ | Months, January to June; $t = 0$ is the end of December | — | — | M0 |
| $\tau$ | Last month of the horizon | — | — | M1 |
| $V$, $i$ | Nodes of the network: plants and warehouses | — | — | M3 |
| $A$, $a$ | Arcs of the network; the arc $a = \langle i, j \rangle$ goes from its origin node $i$ to its destination node $j$ | — | — | M3 |
| $A^{\text{in}}_j$; $A^{\text{out}}_i$ | Arcs that enter node $j$; arcs that leave node $i$ | — | — | M3 |
| $V^d$ | Nodes with demand: the local warehouses | — | — | M3 |
| $c_f$ | Unit assembly cost at plant $f$ | — | €/bike | M0 |
| $K_{ft}$ | Assembly capacity of plant $f$ in month $t$ | — | bikes/month | M0 |
| $d_{it}$ | Orders that node $i$ receives in month $t$ | — | bikes/month | M3 |
| $c^h_i$ | Holding cost at node $i$ | holding cost | €/(bike·month) | M1 |
| $h_{i0}$ | Stock at node $i$ at the end of December | — | bikes | M1 |
| $h^{\text{end}}_i$ | Stock that node $i$ must hold at the end of the horizon | required ending stock | bikes | M1 |
| $K^h_i$ | Units node $i$ can hold | storage capacity | bikes | M2 |
| $c^b_i$ | Cost of one bike backordered for one month at node $i$ | backorder cost | €/(bike·month) | M2 |
| $b_{i0}$; $\bar b_i$ | Backorders at node $i$ at the end of December; maximum at the end of the horizon | backorders | bikes | M2 |
| $c^x_a$ | Cost of carrying one bike along arc $a$ | transport cost | €/bike | M3 |
| $K^x_a$ | Maximum flow on arc $a$ in a month | arc capacity | bikes/month | M3 |
| $K^n_i$ | Maximum flow that can enter node $i$ in a month | node capacity | bikes/month | M3 |
| $p_{ft}$ | Bikes assembled at plant $f$ in month $t$ | — | bikes/month | M0 |
| $h_{it}$ | Bikes in stock at node $i$ at the end of month $t$ | — | bikes | M1 |
| $x_{at}$ | Bikes shipped on arc $a$ in month $t$ | shipments | bikes/month | M1 |
| $s_{it}$ | Bikes that node $i$ delivers in month $t$ to orders of that month or to backorders | deliveries | bikes/month | M2 |
| $b_{it}$ | Orders received by node $i$ and not yet delivered at the end of month $t$ | backorders | bikes | M2 |

The [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/distribution_network.md) explains all the data and results. The places and distances are real; the other data are invented. Running the notebook is optional: it is support material, not part of the assessment.

In [ ]:
# @title Setup { display-mode: "form" }
# In Google Colab, install the example from GitHub (it takes a few seconds).
# On your own computer, install the repository environment instead (see README).
import sys
if "google.colab" in sys.modules:
    url = "git+https://github.com/ula-uab/lscm-decision-making#subdirectory=cases/production_plan"
    %pip install -q {url}
    # A session that already had the example keeps its old code unless it is reinstalled
    %pip install -q --force-reinstall --no-deps {url}
    # and unless its old code is removed from memory, so that the import below reads
    # the new one: no need to restart the session
    import importlib
    for name in [m for m in sys.modules if m == "plants" or m.startswith("plants.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
from plants import distribution_network as dn

## 1. Network, nodes and forecast 4

The first table gives the place of each node. The second is the list of arcs: origin, destination, transport mode, distance, transport cost and arc capacity. The transport cost of an arc is its distance times the transport rate of its mode, given above the table. The third table gives, for each node, its holding cost, its storage capacity, its stock at the end of December, its required ending stock and its node capacity, if it has one. The last table gives the orders of each local warehouse: a share of forecast 4, the same every month.

In [ ]:
# @title Network, nodes and forecast 4 { display-mode: "form" }
dn.show_data()

## 2. Linear model of the distribution network

$$
\begin{aligned}
\min \quad & \sum_{t \in T} \Big( \sum_{f \in F} c_f\, p_{ft} + \sum_{i \in V} c^h_i\, h_{it} + \sum_{a \in A} c^x_a\, x_{at} + \sum_{i \in V^d} c^b_i\, b_{it} \Big) \\
\text{s.t.} \quad & h_{i,t-1} + p_{it} + \sum_{a \in A^{\text{in}}_i} x_{at} = \sum_{a \in A^{\text{out}}_i} x_{at} + s_{it} + h_{it} && \forall i \in V,\ t \in T && \text{(stock balance at each node)} \\
& \sum_{a \in A^{\text{out}}_i} x_{at} + s_{it} \le h_{i,t-1} + p_{it} + \sum_{a \in A^{\text{in}}_i} x_{at} && \forall i \in V,\ t \in T && \text{(shipment limit at each node)} \\
& b_{i,t-1} + d_{it} = s_{it} + b_{it} && \forall i \in V^d,\ t \in T && \text{(backorder balance)} \\
& x_{at} \le K^x_a && \forall a \in A,\ t \in T && \text{(arc capacity)} \\
& \sum_{a \in A^{\text{in}}_i} x_{at} \le K^n_i && \forall i \text{ with a node capacity},\ t \in T && \text{(node capacity)} \\
& h_{it} \le K^h_i && \forall i \in V,\ t \in T && \text{(storage capacity)} \\
& p_{ft} \le K_{ft} && \forall f \in F,\ t \in T && \text{(assembly capacity)} \\
& b_{i\tau} \le \bar b_i \quad \forall i \in V^d, \qquad h_{i\tau} \ge h^{\text{end}}_i \quad \forall i \in V && && \text{(backorders and required ending stock at the end)} \\
& p_{ft},\ h_{it},\ x_{at},\ s_{it},\ b_{it} \ge 0
\end{aligned}
$$

with $h_{i0}$ and $b_{i0}$ given, $p_{it} = 0$ if node $i$ is not a plant and $s_{it} = 0$ if node $i$ has no demand. One stock balance holds at every node: what the node held at the start of the month, plus what it assembles and what arrives on the arcs that enter it, equals what leaves on the arcs that leave it, plus its deliveries, plus what it holds at the end of the month. Only the plants assemble and only the local warehouses deliver; at the rail terminal both are 0, and its storage capacity is 0. There is one flow variable for each arc of the list.

## 3. Optimal plan under forecast 4

The cell first gives the reason why M2 has no solution under forecast 4. Then it solves M3 and gives the bikes assembled at each plant, the flow on each arc, the stock of each node at the end of each month, and the deliveries and backorders of the local warehouses, with the cost of the plan. Below, the **shadow price** of each capacity that is full in some month (€/bike): the change in cost with one more bike of capacity (negative: a saving). The figure on the left draws the network on a map, with the flow on each arc in the month you choose; an arc whose capacity is full is drawn in red, and the rail arc is dashed. The one on the right stacks the stock at each level of the network at the end of each month.

Choose May. By which routes do bikes reach regional warehouse 2, and which arcs are full? Is the rail arc full? In which months is there stock at more than one level of the network, and why then?

In [ ]:
# @title Optimal plan under forecast 4 { display-mode: "form" }
month = "May"  # @param ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
dn.network_plan(month);

## 4. Arc capacity of the rail arc

The slider sets the arc capacity of the rail arc, from the dry port of Zaragoza to the rail terminal in Coslada. The rail terminal holds no stock, so the rail arc carries only what the road arc from the terminal to regional warehouse 2 carries, at most 200 bikes a month. The cell first gives the flows on the three routes to regional warehouse 2 and on the two routes to Valencia, then the plan, with the flows of May on the map.

Lower the arc capacity of the rail arc from 400. From what value does the rail arc fill, and which routes carry the bikes that no longer fit? Below what value is there no solution?

In [ ]:
# @title Optimal plan for an arc capacity of the rail arc { display-mode: "form" }
rail_arc_capacity = 400  # @param {type:"slider", min:0, max:400, step:10}
dn.rail_capacity_test(rail_arc_capacity);

## 5. Transport rate of rail

The slider sets the transport rate of rail, in €/(bike·km); the transport cost of the rail arc is its distance, 336 km, times that rate. The route by rail and the rail terminal competes with the road from the central warehouse to regional warehouse 2, whose transport cost is 3,30 €/bike and whose arc capacity is 150 bikes a month. The cell shows the same tables as the previous one.

Raise the transport rate of rail from 0,006. At what rate do the two routes cost the same, and what happens at exactly that rate? Above it, does the rail arc carry nothing, or what does it carry?

In [ ]:
# @title Optimal plan for a transport rate of rail { display-mode: "form" }
rail_transport_rate = 0.006  # @param {type:"slider", min:0.002, max:0.015, step:0.0005}
dn.rail_rate_test(rail_transport_rate);

## 6. Node capacity of regional warehouse 2

The slider sets the node capacity of regional warehouse 2: the bikes it can receive in a month, by any of its three routes. Regional warehouse 2 supplies Seville alone and shares Valencia with regional warehouse 1. The cell shows the same tables as the previous ones.

Raise the node capacity from 280, then lower it. When is it full, and where do the bikes that it cannot receive go? Below what value is there no solution, and why?

In [ ]:
# @title Optimal plan for a node capacity of regional warehouse 2 { display-mode: "form" }
node_capacity = 280  # @param {type:"slider", min:150, max:400, step:10}
dn.node_capacity_test(node_capacity);

The [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/distribution_network.md) explains all the numbers and has a script that prints them.